# Mini Benchmark Evaluator — Colab demo

Tutorial **"The Science of Benchmarking — What's Measured, What's Missing, What's Next"** (NeurIPS 2025).

We take **two real public benchmarks** (MMLU, GSM8K), run **Gemma-4** models of two sizes plus non-LLM baselines, and apply the tutorial's critical toolkit:

1. **Saturation** — MMLU (knowledge) vs GSM8K (reasoning): where do strong models stop separating? *(GLUE → SuperGLUE)*
2. **Baselines** — random / majority class: is the score measuring competence? *(Measuring what Matters)*
3. **Metric brittleness** — GSM8K naive vs robust grader: right answers a bad metric throws away. *(metric ≠ construct)*
4. **Robustness** — MMLU option-shuffle / distractor twins: accuracy drop under label-preserving noise. *(SuperGLUE)*
5. **Contamination / construct** — discussed in the report.

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Greedy decoding ⇒ deterministic ⇒ reproducible.

## 1. Get the code

In [ ]:
# Clone the demo repo (replace URL if you forked it), then install deps.
REPO_URL = "https://github.com/ThuongHong/science-of-benchmarking-demo.git"
import os, sys, subprocess
REPO_DIR = REPO_URL.rsplit("/", 1)[-1].removesuffix(".git")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--quiet", REPO_URL], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, "src")
!pip install -q -r requirements.txt
print("cwd:", os.getcwd())

## 2. Hugging Face login (Gemma is gated)
Accept the licence once at https://huggingface.co/google/gemma-4-E4B-it , then paste a read token below.

In [ ]:
from huggingface_hub import login
login()  # paste your HF token when prompted

## 3. Build the benchmark subsets (downloads MMLU + GSM8K once)

In [ ]:
import config
from data import build_subsets
subs = build_subsets(config.N_MMLU, config.N_GSM8K, config.SEED)
print({k: len(v) for k, v in subs.items()})
subs["mmlu"][0]

## 4. Run the evaluation (Gemma-4 E2B + E4B + baselines)
First call downloads the models. Cached greedy generations make re-runs instant.

In [ ]:
import evaluate
df = evaluate.run(config.SYSTEMS)
df.to_csv("results/per_item.csv", index=False)
print(len(df), "rows ->", "results/per_item.csv")
df.head()

## 5. Analysis + figures

In [ ]:
import analysis
analysis.main()

In [ ]:
from IPython.display import Image, display
for fig in ["saturation", "metric_gap", "robustness"]:
    display(Image(f"results/figures/{fig}.png"))

## 6. (Optional) add a third, larger rung on Kaggle / A100
The 26B-A4B MoE needs ~14–18 GB and will OOM a single T4. On a bigger GPU, append it to `config.SYSTEMS` and re-run cells 4–5:
```python
config.SYSTEMS.append({"kind": "gemma", "name": "gemma-4-26b",
                        "model_id": "google/gemma-4-26B-A4B-it"})
```